# Continuous Control

---

You are welcome to use this coding environment to train your agent for the project.  Follow the instructions below to get started!

### 0. Clean Up the Zombie Processes
At the end of this notebook, when you execute `env.close()`, it does not clean up the environment completely. Instead, the Unity environment process becomes a "zombie" process. A zombie process is one that has completed execution but still has an entry in the process table because its parent process hasn’t properly reaped it.
You can yourself verify this by running these commands in the terminal. Find the parent process ID (PPID) of the zombie process:
```bash
ps -o pid,ppid,stat,cmd | grep Reacher
```
If the parent process (PPID) is not 1, kill it to clean up the zombie process:
```bash
kill -9 <PPID>
```
Below is the equivalent Python code that checks for and cleans zombie processes using `psutil`. **You need run the cell below only when you restart the Unity environment.** 

> **NOTE**: The code cell below will also kill the Kernel. You should restart it when required.

### 1. Start the Environment

Run the next code cell to install a few packages.  This line will take a few minutes to run!

In [ ]:
!pip -q install .

The environments corresponding to both versions of the environment are already saved in the Workspace and can be accessed at the file paths provided below.  

Please select one of the two options below for loading the environment.

In [ ]:
from unityagents import UnityEnvironment
import numpy as np

# select this option to load version 1 (with a single agent) of the environment
env = UnityEnvironment(file_name='/data/Reacher_One_Linux_NoVis/Reacher_One_Linux_NoVis.x86_64')

# select this option to load version 2 (with 20 agents) of the environment
# env = UnityEnvironment(file_name='/data/Reacher_Linux_NoVis/Reacher.x86_64')

Environments contain **_brains_** which are responsible for deciding the actions of their associated agents. Here we check for the first brain available, and set it as the default brain we will be controlling from Python.

In [ ]:
# get the default brain
brain_name = env.brain_names[0]
brain = env.brains[brain_name]

### 2. Examine the State and Action Spaces

Run the code cell below to print some information about the environment.

In [ ]:
# reset the environment
env_info = env.reset(train_mode=True)[brain_name]

# number of agents
num_agents = len(env_info.agents)
print('Number of agents:', num_agents)

# size of each action
action_size = brain.vector_action_space_size
print('Size of each action:', action_size)

# examine the state space 
states = env_info.vector_observations
state_size = states.shape[1]
print('There are {} agents. Each observes a state with length: {}'.format(states.shape[0], state_size))
print('The state for the first agent looks like:', states[0])

### 3. Take Random Actions in the Environment

In the next code cell, you will learn how to use the Python API to control the agent and receive feedback from the environment.

Note that **in this coding environment, you will not be able to watch the agents while they are training**, and you should set `train_mode=True` to restart the environment.

In [ ]:
env_info = env.reset(train_mode=True)[brain_name]      # reset the environment    
states = env_info.vector_observations                  # get the current state (for each agent)
scores = np.zeros(num_agents)                          # initialize the score (for each agent)
while True:
    actions = np.random.randn(num_agents, action_size) # select an action (for each agent)
    actions = np.clip(actions, -1, 1)                  # all actions between -1 and 1
    env_info = env.step(actions)[brain_name]           # send all actions to tne environment
    next_states = env_info.vector_observations         # get next state (for each agent)
    rewards = env_info.rewards                         # get reward (for each agent)
    dones = env_info.local_done                        # see if episode finished
    scores += env_info.rewards                         # update the score (for each agent)
    states = next_states                               # roll over states to next time step
    if np.any(dones):                                  # exit loop if episode finished
        break
print('Total score (averaged over agents) this episode: {}'.format(np.mean(scores)))

When finished, you can close the environment.

In [ ]:
env.close()

### 4. It's Your Turn!

Train the DDPG agent (defined in `agent.py` + `model.py`) on the 20-arm
Reacher environment. The cell below runs the standard DDPG training
loop using the agent we built; on Udacity's GPU workspace this solves
the environment (rolling-100 mean ≥ +30) in roughly 150–200 episodes.

In [ ]:
import sys
sys.path.append('..')   # so `from agent import DDPGAgent` works inside ./python/

from collections import deque
import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline

from agent import DDPGAgent

# Re-pull the env spec
env_info     = env.reset(train_mode=True)[brain_name]
num_agents   = len(env_info.agents)
state_size   = env_info.vector_observations.shape[1]
action_size  = brain.vector_action_space_size

agent = DDPGAgent(state_size=state_size, action_size=action_size, num_agents=num_agents)

def ddpg(n_episodes=300, target=30.0):
    scores_window = deque(maxlen=100)
    all_scores = []
    for ep in range(1, n_episodes + 1):
        env_info = env.reset(train_mode=True)[brain_name]
        states = env_info.vector_observations
        agent.reset()
        scores = np.zeros(num_agents)
        while True:
            actions = agent.act(states)
            env_info = env.step(actions)[brain_name]
            next_states = env_info.vector_observations
            rewards     = np.array(env_info.rewards)
            dones       = np.array(env_info.local_done, dtype=int)
            agent.step(states, actions, rewards, next_states, dones)
            states = next_states
            scores += rewards
            if dones.any():
                break
        score = float(np.mean(scores))
        scores_window.append(score)
        all_scores.append(score)
        avg = np.mean(scores_window)
        print(f'Episode {ep:3d}\tEpisode score: {score:7.2f}\tRolling-100 mean: {avg:7.2f}')
        if avg >= target and len(scores_window) == 100:
            print(f'\nEnvironment solved in {ep} episodes! Avg over last 100 = {avg:.2f}')
            torch.save(agent.actor_local.state_dict(),  'reacher_actor.pth')
            torch.save(agent.critic_local.state_dict(), 'reacher_critic.pth')
            break
    return all_scores

scores = ddpg(n_episodes=300, target=30.0)

# Plot the per-episode scores + the 100-episode rolling mean.
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(np.arange(1, len(scores) + 1), scores, alpha=0.5, label='Per-episode mean across 20 arms')
rolling = np.convolve(scores, np.ones(100) / 100, mode='valid')
ax.plot(np.arange(100, 100 + len(rolling)), rolling, color='red', label='Rolling-100 mean')
ax.axhline(30.0, color='green', linestyle='--', label='Solved threshold (+30)')
ax.set_xlabel('Episode'); ax.set_ylabel('Score'); ax.set_title('Reacher (20-arm) — DDPG training')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig('reacher_training_curve.png', dpi=120)
plt.show()

np.save('reacher_scores.npy', np.array(scores))